In [ ]:
#!/usr/bin/env python3
"""
GUNDAM Minimal Example - Easy to Understand and Modify
======================================================

This is a stripped-down version focusing on the core concepts:
- 2 true bins, 3 reco bins (small enough to understand)
- 1 systematic parameter (flux)
- Clear step-by-step process

Perfect for learning and experimentation!
"""

import numpy as np
from scipy.optimize import minimize
from scipy.linalg import cholesky

print("="*60)
print("GUNDAM MINIMAL EXAMPLE")
print("="*60)
print()

# ============================================================================
# STEP 1: DEFINE YOUR DATA
# ============================================================================

print("STEP 1: Input Data")
print("-" * 60)

# True bins: momentum ranges (GeV/c)
true_bin_edges = [0.0, 0.5, 1.0]
n_true = 2

# Reco bins: measured momentum ranges (GeV/c)
reco_bin_edges = [0.0, 0.4, 0.7, 1.0]
n_reco = 3

# Response matrix (how truth maps to reco)
# N_ij[i,j] = events from true bin i → reco bin j
N_ij = np.array([
    [2, 1, 0],  # True bin 1: 3 events total
    [0, 2, 1],  # True bin 2: 3 events total
])

print(f"Response matrix N_ij:")
print(N_ij)
print()

# Background per reco bin
B_j = np.array([0, 1, 1])  # Total: 2 background events

print(f"Background: {B_j}")
print()

# Observed data
N_data = np.array([3, 5, 2])  # Total: 10 observed events

print(f"Data: {N_data}")
print()

# True event count before selection (from generator)
N_true_gen = np.array([3, 3])  # We generated 3 events per bin

# Systematic parameter prior: flux uncertainty
flux_prior = 0.0    # Central value
flux_sigma = 0.10   # ±10% uncertainty

print(f"Flux prior: {flux_prior} ± {flux_sigma}")
print()

# ============================================================================
# STEP 2: DEFINE FORWARD MODEL
# ============================================================================

print("STEP 2: Forward Model (Truth → Reco Prediction)")
print("-" * 60)

def forward_model(params):
    """
    Predict reco events given parameters.

    params = [c1, c2, a_flux]
    where:
        c1, c2 = template parameters (scale true bins)
        a_flux = flux systematic parameter
    """
    c1, c2, a_flux = params

    # Template parameters scale each true bin
    c = np.array([c1, c2])

    # Flux weight multiplies all signal
    flux_weight = 1.0 + a_flux

    # Predict reco counts
    N_pred = np.zeros(n_reco)
    for i in range(n_true):
        N_pred += c[i] * flux_weight * N_ij[i, :]

    # Add backgrounds
    N_pred += B_j

    return N_pred

# Test with initial parameters
params_init = np.array([1.0, 1.0, 0.0])  # c1=1, c2=1, flux=0
N_pred_init = forward_model(params_init)

print(f"Initial parameters: c1={params_init[0]}, c2={params_init[1]}, flux={params_init[2]}")
print(f"Initial prediction: {N_pred_init}")
print(f"Data:              {N_data}")
print()

# ============================================================================
# STEP 3: DEFINE LIKELIHOOD
# ============================================================================

print("STEP 3: Likelihood Function")
print("-" * 60)

def neg_log_likelihood(params):
    """
    -2*log(L) = Statistical term + Penalty term
    """
    N_pred = forward_model(params)

    # Statistical: Poisson likelihood
    L_stat = 0.0
    for j in range(n_reco):
        if N_pred[j] > 0 and N_data[j] > 0:
            L_stat += 2 * (N_pred[j] - N_data[j] + N_data[j] * np.log(N_data[j] / N_pred[j]))
        elif N_pred[j] > 0:
            L_stat += 2 * N_pred[j]

    # Penalty: Gaussian prior on flux
    a_flux = params[2]
    L_penalty = ((a_flux - flux_prior) / flux_sigma)**2

    return L_stat + L_penalty

# Initial likelihood
L_init = neg_log_likelihood(params_init)
print(f"Initial -2log(L): {L_init:.2f}")
print()

# ============================================================================
# STEP 4: FIT (MINIMIZE LIKELIHOOD)
# ============================================================================

print("STEP 4: Fitting")
print("-" * 60)

result = minimize(neg_log_likelihood, params_init, method='L-BFGS-B')

params_best = result.x
L_best = result.fun

print(f"Best-fit parameters:")
print(f"  c1 = {params_best[0]:.3f}")
print(f"  c2 = {params_best[1]:.3f}")
print(f"  a_flux = {params_best[2]:.3f}")
print()
print(f"Best-fit -2log(L): {L_best:.2f}")
print(f"Improvement: {L_init - L_best:.2f}")
print()

N_pred_best = forward_model(params_best)
print(f"Best-fit prediction: {N_pred_best}")
print(f"Data:               {N_data}")
print()

# ============================================================================
# STEP 5: COMPUTE UNCERTAINTIES (HESSIAN)
# ============================================================================

print("STEP 5: Computing Uncertainties")
print("-" * 60)

def compute_hessian(params, eps=1e-5):
    """Simple Hessian calculation."""
    n = len(params)
    H = np.zeros((n, n))
    f0 = neg_log_likelihood(params)

    for i in range(n):
        for j in range(i, n):
            if i == j:
                # Diagonal: second derivative
                p_plus = params.copy()
                p_minus = params.copy()
                p_plus[i] += eps
                p_minus[i] -= eps

                f_plus = neg_log_likelihood(p_plus)
                f_minus = neg_log_likelihood(p_minus)

                H[i,i] = (f_plus - 2*f0 + f_minus) / eps**2
            else:
                # Off-diagonal: cross derivative
                p_pp = params.copy()
                p_pm = params.copy()
                p_mp = params.copy()
                p_mm = params.copy()

                p_pp[i] += eps; p_pp[j] += eps
                p_pm[i] += eps; p_pm[j] -= eps
                p_mp[i] -= eps; p_mp[j] += eps
                p_mm[i] -= eps; p_mm[j] -= eps

                H[i,j] = (neg_log_likelihood(p_pp) - neg_log_likelihood(p_pm)
                         - neg_log_likelihood(p_mp) + neg_log_likelihood(p_mm)) / (4*eps**2)
                H[j,i] = H[i,j]

    return H

print("Computing Hessian...")
H = compute_hessian(params_best)
V = np.linalg.inv(H)  # Covariance = inverse Hessian
errors = np.sqrt(np.diag(V))

print(f"Parameter uncertainties:")
print(f"  c1 = {params_best[0]:.3f} ± {errors[0]:.3f}")
print(f"  c2 = {params_best[1]:.3f} ± {errors[1]:.3f}")
print(f"  a_flux = {params_best[2]:.3f} ± {errors[2]:.3f}")
print()

# Correlation matrix
corr = np.zeros((3,3))
for i in range(3):
    for j in range(3):
        corr[i,j] = V[i,j] / (errors[i] * errors[j])

print("Correlation matrix:")
print(corr)
print()

# ============================================================================
# STEP 6: EXTRACT CROSS SECTION
# ============================================================================

print("STEP 6: Cross Section Extraction")
print("-" * 60)

# Normalization (made-up values for demonstration)
Phi = 1.5e13           # flux (ν/cm²)
N_target = 5.0e29      # targets (nucleons)
Delta_p = np.array([0.5, 0.5])  # bin widths (GeV/c)

# Extract signal from template parameters
c1, c2, a_flux = params_best
N_signal = np.array([c1, c2]) * N_true_gen * (1 + a_flux)

print(f"Signal events (truth level):")
print(f"  Bin 1: {N_signal[0]:.2f}")
print(f"  Bin 2: {N_signal[1]:.2f}")
print()

# Cross sections
sigma = N_signal / (Phi * N_target * Delta_p)

print(f"Cross sections:")
print(f"  Bin 1: {sigma[0]:.2e} cm²/GeV/nucleon")
print(f"  Bin 2: {sigma[1]:.2e} cm²/GeV/nucleon")
print()

# ============================================================================
# STEP 7: TOY THROWING
# ============================================================================

print("STEP 7: Toy Throwing for Uncertainties")
print("-" * 60)

n_toys = 1000
print(f"Throwing {n_toys} toys...")

# Cholesky decomposition
L = cholesky(V, lower=True)

# Storage
sigma_toys = np.zeros((n_toys, n_true))

for k in range(n_toys):
    # Generate correlated parameters
    z = np.random.randn(3)
    params_toy = params_best + L @ z

    # Extract cross section
    c1_toy, c2_toy, a_flux_toy = params_toy
    N_signal_toy = np.array([c1_toy, c2_toy]) * N_true_gen * (1 + a_flux_toy)
    sigma_toy = N_signal_toy / (Phi * N_target * Delta_p)

    sigma_toys[k, :] = sigma_toy

# Statistics
sigma_mean = np.mean(sigma_toys, axis=0)
sigma_std = np.std(sigma_toys, axis=0)

print("Done!")
print()
print("Final Results with Uncertainties:")
print(f"  σ₁ = {sigma_mean[0]:.2e} ± {sigma_std[0]:.2e} cm²/GeV/nucleon")
print(f"  σ₂ = {sigma_mean[1]:.2e} ± {sigma_std[1]:.2e} cm²/GeV/nucleon")
print()

# ============================================================================
# INTERPRETATION
# ============================================================================

print("="*60)
print("INTERPRETATION")
print("="*60)
print()

print(f"Template parameter c₁ = {params_best[0]:.2f}:")
print(f"  → True cross section in bin 1 is {(params_best[0]-1)*100:+.0f}% different from MC")
print()

print(f"Template parameter c₂ = {params_best[1]:.2f}:")
print(f"  → True cross section in bin 2 is {(params_best[1]-1)*100:+.0f}% different from MC")
print()

print(f"Flux parameter a_flux = {params_best[2]:.3f}:")
print(f"  → Flux is {params_best[2]*100:+.1f}% different from prior")
print(f"  → This is a {abs(params_best[2])/flux_sigma:.1f}σ pull")
print()

print("Key insight:")
print("  Templates freely adjusted to match data (no prior constraints)")
print("  Flux stayed near prior (strong 10% constraint)")
print("  → Model-independent cross section measurement!")
print()

# ============================================================================
# EXPERIMENTS TO TRY
# ============================================================================

print("="*60)
print("EXPERIMENTS TO TRY")
print("="*60)
print()

print("1. Change the data:")
print("   N_data = np.array([4, 6, 3])  # More excess")
print("   → See how template parameters respond")
print()

print("2. Tighten flux constraint:")
print("   flux_sigma = 0.05  # ±5% instead of ±10%")
print("   → Flux pulls less, templates adjust more")
print()

print("3. Remove flux constraint:")
print("   flux_sigma = 100.0  # Essentially no constraint")
print("   → Flux and templates become degenerate!")
print()

print("4. Add more systematics:")
print("   params = [c1, c2, a_flux, a_detector]")
print("   → Study correlations")
print()

print("5. Generate fake data from best-fit:")
print("   N_fake = forward_model(params_best).astype(int)")
print("   → Closure test: should recover parameters")
print()

print("Now modify this script and experiment!")
print("="*60)
